# RQ1: Binary Classification — Can ML Predict High-ROI Campaigns?

**Research Question:** Can machine learning models accurately predict whether a marketing campaign achieves above-median ROI, and which campaign features are most predictive of conversion success?

**Task:** Binary Classification  
**Target:** `High_ROI` (1 if ROI ≥ median ROI, else 0)  
**Models:** Logistic Regression, Random Forest, XGBoost  
**Dataset:** Marketing and Product Performance Dataset (Kaggle)

In [41]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, auc, confusion_matrix,
    ConfusionMatrixDisplay, cohen_kappa_score, classification_report
)
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

# ── Style constants ──────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800']
sns.set_palette(PALETTE)
RANDOM_STATE = 42
OUTPUT_DIR = '/kaggle/working/'

def save_figure(fig, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    fig.savefig(path, format='pdf', bbox_inches='tight', dpi=300)
    plt.close(fig)
    print(f'Saved figure: {path}')

def save_table(df, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    print(f'Saved table:  {path}')

print('Imports OK')

Imports OK


## 1. Data Loading

In [42]:
input_dir = '/kaggle/input'
data_files = [
    os.path.join(root, f)
    for root, dirs, files in os.walk(input_dir)
    for f in files if f.endswith('.xlsx') or f.endswith('.xls') or f.endswith('.csv')
]
print('Found files:', data_files)
FILE_PATH = data_files[0]

df = pd.read_csv(FILE_PATH) if FILE_PATH.endswith('.csv') else pd.read_excel(FILE_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

Found files: ['/kaggle/input/datasets/vanishjr/marketing-product-performance/marketing_and_product_performance.csv']
Shape: (10000, 17)
Columns: ['Campaign_ID', 'Product_ID', 'Budget', 'Clicks', 'Conversions', 'Revenue_Generated', 'ROI', 'Customer_ID', 'Subscription_Tier', 'Subscription_Length', 'Flash_Sale_ID', 'Discount_Level', 'Units_Sold', 'Bundle_ID', 'Bundle_Price', 'Customer_Satisfaction_Post_Refund', 'Common_Keywords']


,Campaign_ID,Product_ID,Budget,Clicks,Conversions,Revenue_Generated,ROI,Customer_ID,Subscription_Tier,Subscription_Length,Flash_Sale_ID,Discount_Level,Units_Sold,Bundle_ID,Bundle_Price,Customer_Satisfaction_Post_Refund,Common_Keywords
0,CMP_RLSDVN,PROD_HBJFA3,41770.45,4946,73,15520.09,1.94,CUST_1K7G39,Premium,4,FLASH_1VFK5K,43,34,BNDL_29U6W5,433.80,4,Affordable
1,CMP_JHHUE9,PROD_OE8YNJ,29900.93,570,510,30866.17,0.76,CUST_0DWS6F,Premium,4,FLASH_1M6COK,28,97,BNDL_ULV60J,289.29,2,Innovative
2,CMP_6SBOWN,PROD_4V8A08,22367.45,3546,265,32585.62,1.41,CUST_BR2GST,Basic,9,FLASH_J4PEON,51,160,BNDL_0HY0EF,462.87,4,Affordable
3,CMP_Q31QCU,PROD_A1Q6ZB,29957.54,2573,781,95740.12,3.32,CUST_6TBY6K,Premium,32,FLASH_1TOVXT,36,159,BNDL_AI09BC,334.16,1,Durable
4,CMP_AY0UTJ,PROD_F57N66,36277.19,818,79,81990.43,3.53,CUST_XASI45,Standard,29,FLASH_AOBHXL,20,52,BNDL_R03ITT,371.67,2,Affordable


In [43]:
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
df.describe()

Campaign_ID                           object
Product_ID                            object
Budget                               float64
Clicks                                 int64
Conversions                            int64
Revenue_Generated                    float64
ROI                                  float64
Customer_ID                           object
Subscription_Tier                     object
Subscription_Length                    int64
Flash_Sale_ID                         object
Discount_Level                         int64
Units_Sold                             int64
Bundle_ID                             object
Bundle_Price                         float64
Customer_Satisfaction_Post_Refund      int64
Common_Keywords                       object
dtype: object

Missing values:
Campaign_ID                          0
Product_ID                           0
Budget                               0
Clicks                               0
Conversions                          0
Revenue_G

,Budget,Clicks,Conversions,Revenue_Generated,ROI,Subscription_Length,Discount_Level,Units_Sold,Bundle_Price,Customer_Satisfaction_Post_Refund
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,25263.607524,2481.903600,498.978700,50038.627579,2.756365,18.016500,39.421000,100.689600,275.198346,2.500900
std,14350.085927,1435.973623,289.479495,28545.702337,1.296901,10.149666,17.212925,57.074015,129.218710,1.113249
min,500.440000,10.000000,1.000000,1002.080000,0.500000,1.000000,10.000000,1.000000,50.010000,1.000000
25%,12789.190000,1225.750000,247.000000,25264.255000,1.630000,9.000000,24.000000,52.000000,165.717500,2.000000
50%,25030.170000,2451.000000,499.000000,49513.815000,2.750000,18.000000,39.000000,101.000000,272.555000,2.000000
75%,37921.725000,3723.000000,751.000000,74507.157500,3.890000,27.000000,54.000000,150.000000,387.322500,3.000000
max,49999.630000,4999.000000,999.000000,99999.470000,5.000000,35.000000,69.000000,199.000000,499.970000,4.000000


## 2. Target Variable Engineering

In [44]:
median_roi = df['ROI'].median()
df['High_ROI'] = (df['ROI'] >= median_roi).astype(int)
print(f'Median ROI: {median_roi:.4f}')
print('Class distribution:')
print(df['High_ROI'].value_counts())
print(df['High_ROI'].value_counts(normalize=True).round(3))

Median ROI: 2.7500
Class distribution:
High_ROI
1    5019
0    4981
Name: count, dtype: int64
High_ROI
1    0.502
0    0.498
Name: proportion, dtype: float64


## 3. Feature Engineering & Preprocessing

In [45]:
# Binary flags from ID columns
df['Has_Flash_Sale'] = df['Flash_Sale_ID'].notna().astype(int)
df['Has_Bundle']     = df['Bundle_ID'].notna().astype(int)

# Rate features (avoid division by zero)
df['Conversion_Rate']    = np.where(df['Clicks'] > 0, df['Conversions'] / df['Clicks'], 0)
df['Cost_Per_Conversion'] = np.where(df['Conversions'] > 0, df['Budget'] / df['Conversions'], 0)

# Drop leakage columns and raw IDs
DROP_COLS = ['Campaign_ID', 'Product_ID', 'Customer_ID', 'Flash_Sale_ID', 'Bundle_ID',
             'ROI', 'Revenue_Generated', 'High_ROI']  # ROI & Revenue leak the target

NUMERIC_FEATURES = [
    'Budget', 'Clicks', 'Conversions', 'Discount_Level', 'Units_Sold',
    'Bundle_Price', 'Subscription_Length', 'Has_Flash_Sale', 'Has_Bundle',
    'Conversion_Rate', 'Cost_Per_Conversion'
]
CAT_FEATURES = ['Subscription_Tier']

X = df[NUMERIC_FEATURES + CAT_FEATURES].copy()
y = df['High_ROI'].copy()

print(f'Features: {X.shape[1]} | Samples: {X.shape[0]}')
X.head()

Features: 12 | Samples: 10000


,Budget,Clicks,Conversions,Discount_Level,Units_Sold,Bundle_Price,Subscription_Length,Has_Flash_Sale,Has_Bundle,Conversion_Rate,Cost_Per_Conversion,Subscription_Tier
0,41770.45,4946,73,43,34,433.80,4,1,1,0.014759,572.197945,Premium
1,29900.93,570,510,28,97,289.29,4,1,1,0.894737,58.629275,Premium
2,22367.45,3546,265,51,160,462.87,9,1,1,0.074732,84.405472,Basic
3,29957.54,2573,781,36,159,334.16,32,1,1,0.303537,38.357926,Premium
4,36277.19,818,79,20,52,371.67,29,1,1,0.096577,459.204937,Standard


## 4. Exploratory Data Analysis

In [46]:
# Class distribution bar
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Class distribution
df['High_ROI'].value_counts().plot(kind='bar', ax=axes[0], color=PALETTE[:2], edgecolor='white')
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('High ROI (0=Low, 1=High)')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Low ROI', 'High ROI'], rotation=0)

# Panel 2: Budget by High_ROI
sns.boxplot(data=df, x='High_ROI', y='Budget', ax=axes[1], palette=PALETTE[:2])
axes[1].set_title('Budget by ROI Class', fontsize=13, fontweight='bold')
axes[1].set_xticklabels(['Low ROI', 'High ROI'])

# Panel 3: Conversion Rate by High_ROI
sns.boxplot(data=df, x='High_ROI', y='Conversion_Rate', ax=axes[2], palette=PALETTE[:2])
axes[2].set_title('Conversion Rate by ROI Class', fontsize=13, fontweight='bold')
axes[2].set_xticklabels(['Low ROI', 'High ROI'])

fig.suptitle('RQ1 — Exploratory Data Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'rq1_feature_boxplots.pdf')
plt.show()

Saved figure: /kaggle/working/rq1_feature_boxplots.pdf


In [47]:
# Correlation heatmap
corr_cols = NUMERIC_FEATURES + ['High_ROI']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            ax=ax, linewidths=0.5, mask=False, annot_kws={'size': 8})
ax.set_title('Feature Correlation Heatmap (RQ1)', fontsize=14, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'rq1_correlation_heatmap.pdf')
plt.show()

Saved figure: /kaggle/working/rq1_correlation_heatmap.pdf


## 5. Preprocessing Pipeline

In [48]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, NUMERIC_FEATURES),
    ('cat', cat_transformer, CAT_FEATURES)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Train class balance: {y_train.value_counts(normalize=True).round(3).to_dict()}')

Train: (8000, 12) | Test: (2000, 12)
Train class balance: {1: 0.502, 0: 0.498}


## 6. Model Training & Cross-Validation

In [49]:
models = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost':             XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6,
                                         random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scoring = ['accuracy', 'f1', 'roc_auc']

results = []
fitted_models = {}

for name, clf in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', clf)])
    t0 = time.time()
    cv_res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=cv_scoring, return_train_score=False)
    pipe.fit(X_train, y_train)
    train_time = round(time.time() - t0, 2)
    fitted_models[name] = pipe

    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    results.append({
        'Model':         name,
        'Accuracy':      round(accuracy_score(y_test, y_pred), 4),
        'Precision':     round(precision_score(y_test, y_pred), 4),
        'Recall':        round(recall_score(y_test, y_pred), 4),
        'F1':            round(f1_score(y_test, y_pred), 4),
        'ROC_AUC':       round(roc_auc_score(y_test, y_proba), 4),
        'Kappa':         round(cohen_kappa_score(y_test, y_pred), 4),
        'CV_F1_Mean':    round(cv_res['test_f1'].mean(), 4),
        'CV_AUC_Mean':   round(cv_res['test_roc_auc'].mean(), 4),
        'Train_Time_s':  train_time
    })
    print(f'{name}: AUC={results[-1]["ROC_AUC"]}  F1={results[-1]["F1"]}  Kappa={results[-1]["Kappa"]}')

results_df = pd.DataFrame(results)
results_df

Logistic Regression: AUC=0.517  F1=0.5104  Kappa=0.0079
Random Forest: AUC=0.5104  F1=0.5262  Kappa=0.0127
XGBoost: AUC=0.4983  F1=0.5055  Kappa=0.008


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,Kappa,CV_F1_Mean,CV_AUC_Mean,Train_Time_s
0,Logistic Regression,0.5040,0.5059,0.5149,0.5104,0.5170,0.0079,0.5147,0.5078,0.41
1,Random Forest,0.5065,0.5079,0.5458,0.5262,0.5104,0.0127,0.5220,0.5046,9.36
2,XGBoost,0.5040,0.5060,0.5050,0.5055,0.4983,0.0080,0.5113,0.5043,2.59


In [50]:
save_table(results_df, 'rq1_model_comparison.csv')

Saved table:  /kaggle/working/rq1_model_comparison.csv


## 7. Best Model Evaluation

In [51]:
best_name = results_df.loc[results_df['ROC_AUC'].idxmax(), 'Model']
best_pipe = fitted_models[best_name]
print(f'Best model: {best_name}')
print(classification_report(y_test, best_pipe.predict(X_test), target_names=['Low ROI', 'High ROI']))

Best model: Logistic Regression
              precision    recall  f1-score   support

     Low ROI       0.50      0.49      0.50       996
    High ROI       0.51      0.51      0.51      1004

    accuracy                           0.50      2000
   macro avg       0.50      0.50      0.50      2000
weighted avg       0.50      0.50      0.50      2000



## 8. Publication-Ready Figures

In [52]:
# ── ROC Curves (all 3 models) ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

for (name, pipe), color in zip(fitted_models.items(), PALETTE):
    y_proba = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})', linewidth=2, color=color)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — High-ROI Campaign Prediction (RQ1)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout()
save_figure(fig, 'rq1_roc_curves.pdf')
plt.show()

Saved figure: /kaggle/working/rq1_roc_curves.pdf


In [53]:
# ── Confusion Matrix (best model) ───────────────────────────────────────────
y_pred_best = best_pipe.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Low ROI', 'High ROI'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_name} (RQ1)', fontsize=13, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'rq1_confusion_matrix_best_model.pdf')
plt.show()

Saved figure: /kaggle/working/rq1_confusion_matrix_best_model.pdf


## 9. Conclusions

**Summary of findings:**

In [54]:
best_row = results_df.loc[results_df['ROC_AUC'].idxmax()]
print('=' * 60)
print('RQ1 CONCLUSION')
print('=' * 60)
print(f'Best model: {best_row["Model"]}')
print(f'  ROC-AUC : {best_row["ROC_AUC"]}')
print(f'  F1 Score: {best_row["F1"]}')
print(f'  Kappa   : {best_row["Kappa"]}')
print()
print('Outputs saved:')
print('  rq1_roc_curves.pdf')
print('  rq1_confusion_matrix_best_model.pdf')
print('  rq1_feature_boxplots.pdf')
print('  rq1_correlation_heatmap.pdf')
print('  rq1_model_comparison.csv')

RQ1 CONCLUSION
Best model: Logistic Regression
  ROC-AUC : 0.517
  F1 Score: 0.5104
  Kappa   : 0.0079

Outputs saved:
  rq1_roc_curves.pdf
  rq1_confusion_matrix_best_model.pdf
  rq1_feature_boxplots.pdf
  rq1_correlation_heatmap.pdf
  rq1_model_comparison.csv
